#Fast Fourier Transforms
Project #2 for AM 148 Spring 2026

---


Author: Dhani Ram II

Last Updated: Monday, 08 June 2026


## Cell 1: Environment Setup

In [1]:
!pip install torch>=2.4 triton>=3.6

import torch
import triton

print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
print(f"Torch version: {torch.__version__}")
print(f"Triton version: {triton.__version__}")

ERROR: Operation cancelled by user
GPU Available: True
GPU Name: Tesla T4
Torch version: 2.11.0+cu128
Triton version: 3.6.0


## Cell 2: Clone Files from GitHub

In [2]:
!git clone https://github.com/ucsc-am148/fast-fourier-transforms-RamTheHedgehog
%cd fast-fourier-transforms-RamTheHedgehog
!ls

Cloning into 'fast-fourier-transforms-RamTheHedgehog'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 48 (delta 18), reused 15 (delta 15), pack-reused 23 (from 1)
Receiving objects: 100% (48/48), 6.97 MiB | 28.22 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/fast-fourier-transforms-RamTheHedgehog
 ALGORITHMS.md		  kernels_golden.py   sanity_check.py
'FFT Documentation.pdf'   kernels.py	      twiddle_check.py
 golden.pt		  README.md	      twiddles.py
 harness.py		  requirements.txt    twiddles_ref.pyc


## Cell 3: Run and Excuite the Twiddles.py Python File


In [3]:
%%writefile twiddles.py
import math
import torch


# =============================================================================
# Pattern 1: radix-2 length-N/2 twiddles  (F2, F3)
# =============================================================================

def make_radix2_twiddles(
    N: int,
    dtype: torch.dtype = torch.float32,
    device: str = 'cuda',
) -> tuple[torch.Tensor, torch.Tensor]:
    """w_N^k for k in [0, N/2). Returns (tw_re, tw_im), each shape (N//2,).

    Used by the radix-2 butterfly: stage s reads twiddle at index
    (k & (2**s - 1)) * (N >> (s+1)), so the table only needs the lower half
    of one full period."""
    k = torch.arange(N // 2, device=device)
    angle = -2.0 * math.pi * k / N
    tw_re = torch.cos(angle).to(dtype)
    tw_im = torch.sin(angle).to(dtype)
    return tw_re, tw_im


# =============================================================================
# Pattern 2: per-stage radix-16 twiddles  (F4; reused by F5/F6/F7 via F4)
# =============================================================================

def _column_axis_labeling(L: int) -> list[tuple]:
    """Track axis labels through the per-stage permute schedule.

    Convention: input n decomposes as n = sum_i d_i * 16^(L-1-i) with d_0 the
    high digit; output k similarly with e_i. Initial tile has axis i labeled
    ('d', i). At each stage s the kernel applies perm = (s,) + (others in
    original order), bringing axis s to position 0; the four-tl.dot then
    transforms position 0 from ('d', s) to ('e', L-1-s).

    Returns a list of length L; entry s is the tuple of L-1 labels at axis
    positions 1..L-1 of the (16,)*L tile *after* the stage-s permute.
    """
    A = [('d', i) for i in range(L)]
    out = []
    for s in range(L):
        P = [A[s]] + [A[i] for i in range(L) if i != s]
        out.append(tuple(P[1:]))
        A = [('e', L - 1 - s)] + P[1:]
    return out


def make_radix16_twiddles(
    N: int,
    device: str = 'cuda',
) -> tuple[torch.Tensor, torch.Tensor]:
    """Per-stage radix-16 Cooley-Tukey twiddles, stacked. Returns
    (tw_re, tw_im), each shape (L, 16, N//16) fp16. L = log_16(N).

    Stage-0 slice is ones (kernel skips the multiply on s == 0). Stage s > 0
    is built from the labeling above via:
        tw[m, c] = exp(-2*pi*i * m * t / 16^(s+1))
        t = sum_{j=0}^{s-1} e_{L-1-j}_value(c) * 16^j
    where e_{L-1-j}_value(c) reads the base-16 digit of c at the position
    given by _column_axis_labeling(L)[s].
    """
    L = int(math.log(N, 16))
    tw_re = torch.ones((L, 16, N // 16), dtype=torch.float16, device=device)
    tw_im = torch.zeros((L, 16, N // 16), dtype=torch.float16, device=device)

    if L <= 1:
        return tw_re, tw_im

    c = torch.arange(N // 16, device=device)
    m = torch.arange(16, device=device).unsqueeze(1)
    labels = _column_axis_labeling(L)

    for s in range(1, L):
        axes = labels[s]
        t = torch.zeros(N // 16, device=device, dtype=torch.float32)

        for j in range(s):
            target_label = ('e', L - 1 - j)
            idx = axes.index(target_label)


            val = (c // (16 ** (L - 2 - idx))) % 16
            t += val * (16 ** j)

        t = t.unsqueeze(0)
        angle = -2.0 * math.pi * m * t / (16 ** (s + 1))
        tw_re[s] = torch.cos(angle).to(torch.float16)
        tw_im[s] = torch.sin(angle).to(torch.float16)

    return tw_re, tw_im


# =============================================================================
# Pattern 3: Bailey cross-term twiddles  (F3, F5, F6, F7)
# =============================================================================

def make_bailey_cross_twiddles(
    m0: int,
    M: int,
    N: int,
    dtype: torch.dtype = torch.float16,
    device: str = 'cuda',
) -> tuple[torch.Tensor, torch.Tensor]:
    """w_N^{n1 * kM} for n1 in [0, m0), kM in [0, M). Returns (re, im), each
    shape (m0, M).

    F3 calls this with dtype=torch.float32 (the radix-2 tier is fp32);
    F5/F6/F7 call it with dtype=torch.float16 (the tcFFT tier is fp16). The
    Bailey identity holds for any N >= m0 * M; in practice N == m0 * M.
    """
    n1 = torch.arange(m0, device=device).unsqueeze(1)
    kM = torch.arange(M, device=device).unsqueeze(0)
    angle = -2.0 * math.pi * n1 * kM / N
    tw_re = torch.cos(angle).to(dtype)
    tw_im = torch.sin(angle).to(dtype)
    return tw_re, tw_im


# =============================================================================
# Scaffolding tables
# =============================================================================

def make_dft_matrix(
    N: int,
    dtype: torch.dtype = torch.float16,
    device: str = 'cuda',
) -> tuple[torch.Tensor, torch.Tensor]:
    """Full (N, N) DFT matrix. Returns (W_re, W_im).

    W[j, k] = exp(-2*pi*i * j * k / N). Used by F1 (DFT-as-complex-matmul).
    """
    j = torch.arange(N, device=device).unsqueeze(1)
    k = torch.arange(N, device=device).unsqueeze(0)
    angle = -2.0 * math.pi * j * k / N
    W_re = torch.cos(angle).to(dtype)
    W_im = torch.sin(angle).to(dtype)
    return W_re, W_im


def make_dft_R_padded(
    R: int,
    device: str = 'cuda',
) -> tuple[torch.Tensor, torch.Tensor]:
    """Length-R DFT padded to (16, 16) fp16. Returns (M_re, M_im).

    Pad the length-R row to 16 with zeros, hit it with a (16, 16) matrix whose
    first R columns are F_R (rows wrap mod R), take the first R output rows.
    This makes the >=16x16 tl.dot requirement hold for all R in {2, 4, 8, 16}.
    """
    r = torch.arange(16, device=device).unsqueeze(1)
    c = torch.arange(16, device=device).unsqueeze(0)

    angle = -2.0 * math.pi * (r % R) * c / R
    M_re = torch.cos(angle)
    M_im = torch.sin(angle)


    M_re[:, R:] = 0.0
    M_im[:, R:] = 0.0

    return M_re.to(torch.float16), M_im.to(torch.float16)


def bit_reversal_perm(N: int, device: str = 'cuda') -> torch.Tensor:
    """Length-N bit-reversal permutation as a (N,) int32 tensor.

    rev[i] is the integer whose n_bits=log2(N) binary representation is i's
    bits in reversed order.
    """
    n_bits = int(math.log2(N))
    rev = torch.zeros(N, dtype=torch.int32, device=device)
    idx = torch.arange(N, dtype=torch.int32, device=device)


    for i in range(n_bits):
        rev |= ((idx >> i) & 1) << (n_bits - 1 - i)

    return rev

Overwriting twiddles.py


## Cell 4: Runs the new version of the kernels.py Python File. Updates by me

In [4]:
%%writefile kernels.py
import math

import torch
import triton
import triton.language as tl


# Tunings -- GIVEN.
F4_L2_BLOCK_B = 2
DFT_BLOCK_B = 16
SCALE_BLOCK = 32
TRANSPOSE_BLOCK = 32


# =============================================================================
# Device-function helper: complex matmul
# =============================================================================

@triton.jit
def _cdot(a_re, a_im, b_re, b_im):
    """Complex matmul Y = A @ B as four real tl.dot calls."""
    y_re = tl.dot(a_re, b_re, out_dtype=tl.float32) - tl.dot(a_im, b_im, out_dtype=tl.float32)
    y_im = tl.dot(a_re, b_im, out_dtype=tl.float32) + tl.dot(a_im, b_re, out_dtype=tl.float32)
    return y_re, y_im


# =============================================================================
# Chunk factorization for F6 / F7
# =============================================================================

def f6_factor(N: int) -> list[int]:
    """Factor N = 2^k into FFT chunks."""
    chunks = []
    while N > 1:
        if N % 256 == 0:
            chunks.append(256)
            N //= 256
        elif N % 16 == 0:
            chunks.append(16)
            N //= 16
        else:
            chunks.append(N)
            N = 1
    return chunks

f7_factor = f6_factor


# =============================================================================
# F1: DFT as one dense complex matmul (four tl.dot)
# =============================================================================

@triton.jit
def f1_kernel(
    x_re_ptr, x_im_ptr,
    W_re_ptr, W_im_ptr,
    y_re_ptr, y_im_ptr,
    B,
    N: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_K: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)

    acc_re = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    acc_im = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    for k in range(0, N, BLOCK_K):
        k_offs = k + tl.arange(0, BLOCK_K)

        x_offs = offs_m[:, None] * N + k_offs[None, :]
        w_offs = offs_n[None, :] * N + k_offs[:, None]

        x_mask = (offs_m[:, None] < B) & (k_offs[None, :] < N)
        w_mask = (offs_n[None, :] < N) & (k_offs[:, None] < N)

        x_re = tl.load(x_re_ptr + x_offs, mask=x_mask, other=0.0)
        x_im = tl.load(x_im_ptr + x_offs, mask=x_mask, other=0.0)

        w_re = tl.load(W_re_ptr + w_offs, mask=w_mask, other=0.0)
        w_im = tl.load(W_im_ptr + w_offs, mask=w_mask, other=0.0)

        dot_re, dot_im = _cdot(x_re, x_im, w_re, w_im)
        acc_re += dot_re
        acc_im += dot_im

    y_offs = offs_m[:, None] * N + offs_n[None, :]
    y_mask = (offs_m[:, None] < B) & (offs_n[None, :] < N)

    tl.store(y_re_ptr + y_offs, acc_re, mask=y_mask)
    tl.store(y_im_ptr + y_offs, acc_im, mask=y_mask)


def f1_launch(x_re, x_im, W_re, W_im, y_re, y_im):
    N = W_re.shape[0]
    B = x_re.numel() // N

    BLOCK_M, BLOCK_N, BLOCK_K = 16, 16, 32
    grid = (triton.cdiv(B, BLOCK_M), triton.cdiv(N, BLOCK_N))
    f1_kernel[grid](
        x_re, x_im, W_re, W_im, y_re, y_im, B, N,
        BLOCK_M=BLOCK_M, BLOCK_K=BLOCK_K, BLOCK_N=BLOCK_N
    )


# =============================================================================
# F2: radix-2 Cooley-Tukey, single program per signal
# =============================================================================

@triton.jit
def f2_kernel(
    x_re_ptr, x_im_ptr,
    y_re_ptr, y_im_ptr,
    tw_re_ptr, tw_im_ptr,
    perm_ptr,
    bt_re_ptr, bt_im_ptr,
    OUTER_DIM, N_TOTAL,
    N: tl.constexpr,
    LOG2_N: tl.constexpr,
    BAILEY_EPILOGUE: tl.constexpr,
    STRIDED_STORE: tl.constexpr,
):
    pid = tl.program_id(0)

    offs = tl.arange(0, N)
    rev_offs = tl.load(perm_ptr + offs)


    in_offs = pid * N + rev_offs
    v_re = tl.load(x_re_ptr + in_offs)
    v_im = tl.load(x_im_ptr + in_offs)


    for s in tl.static_range(LOG2_N):
        half_step = 1 << s

        tw_idx = (offs & (half_step - 1)) * (N >> (s + 1))
        w_re = tl.load(tw_re_ptr + tw_idx)
        w_im = tl.load(tw_im_ptr + tw_idx)

        partner = offs ^ half_step
        partner_re = tl.gather(v_re, partner, axis=0)
        partner_im = tl.gather(v_im, partner, axis=0)

        is_top = (offs & half_step) == 0


        bot_re = tl.where(is_top, partner_re, v_re)
        bot_im = tl.where(is_top, partner_im, v_im)

        tw_mul_re = bot_re * w_re - bot_im * w_im
        tw_mul_im = bot_re * w_im + bot_im * w_re

        top_re = tl.where(is_top, v_re, partner_re)
        top_im = tl.where(is_top, v_im, partner_im)


        v_re = tl.where(is_top, top_re + tw_mul_re, top_re - tw_mul_re)
        v_im = tl.where(is_top, top_im + tw_mul_im, top_im - tw_mul_im)

    if BAILEY_EPILOGUE:
        n1 = pid % OUTER_DIM
        bt_offs = n1 * N + offs
        b_re = tl.load(bt_re_ptr + bt_offs)
        b_im = tl.load(bt_im_ptr + bt_offs)

        scaled_re = v_re * b_re - v_im * b_im
        scaled_im = v_re * b_im + v_im * b_re
        v_re, v_im = scaled_re, scaled_im

    if STRIDED_STORE:
        b = pid // OUTER_DIM
        k2 = pid % OUTER_DIM
        out_offs = b * N_TOTAL + offs * OUTER_DIM + k2
    else:
        out_offs = pid * N + offs

    tl.store(y_re_ptr + out_offs, v_re)
    tl.store(y_im_ptr + out_offs, v_im)


def f2_launch(x_re, x_im, y_re, y_im, tw_re, tw_im, perm):
    N = perm.shape[0]
    B = x_re.numel() // N
    LOG2_N = int(math.log2(N))
    grid = (B,)
    f2_kernel[grid](
        x_re, x_im, y_re, y_im,
        tw_re, tw_im, perm,
        tw_re, tw_im,
        1, 0, N, LOG2_N,
        BAILEY_EPILOGUE=False, STRIDED_STORE=False
    )


# =============================================================================
# transpose_kernel: (B, R, C) -> (B, C, R), paired re/im
# =============================================================================

@triton.jit
def transpose_kernel(
    x_re_ptr, x_im_ptr,
    y_re_ptr, y_im_ptr,
    R, C,
    BLOCK_R: tl.constexpr,
    BLOCK_C: tl.constexpr,
):
    pid_r = tl.program_id(0)
    pid_c = tl.program_id(1)
    pid_b = tl.program_id(2)

    offs_r = pid_r * BLOCK_R + tl.arange(0, BLOCK_R)
    offs_c = pid_c * BLOCK_C + tl.arange(0, BLOCK_C)

    in_offs = pid_b * (R * C) + offs_r[:, None] * C + offs_c[None, :]
    out_offs = pid_b * (R * C) + offs_c[None, :] * R + offs_r[:, None]

    mask = (offs_r[:, None] < R) & (offs_c[None, :] < C)

    v_re = tl.load(x_re_ptr + in_offs, mask=mask)
    v_im = tl.load(x_im_ptr + in_offs, mask=mask)

    tl.store(y_re_ptr + out_offs, v_re, mask=mask)
    tl.store(y_im_ptr + out_offs, v_im, mask=mask)


# =============================================================================
# F4: tcFFT radix-16 single-program FFT (N = 256, L = 2)
# =============================================================================

@triton.jit
def f4_kernel_L2(
    x_re_ptr, x_im_ptr,
    y_re_ptr, y_im_ptr,
    F_re_ptr, F_im_ptr,
    tw_re_ptr, tw_im_ptr,
    B, M,
    BLOCK_B: tl.constexpr,
    STAGE_STOP: tl.constexpr,
    STORE_T: tl.constexpr,
):
    pid = tl.program_id(0)
    offs_b = pid * BLOCK_B + tl.arange(0, BLOCK_B)
    offs_n = tl.arange(0, 256)

    in_offs = offs_b[:, None] * 256 + offs_n[None, :]
    mask_b = offs_b[:, None] < B

    x_re = tl.load(x_re_ptr + in_offs, mask=mask_b, other=0.0)
    x_im = tl.load(x_im_ptr + in_offs, mask=mask_b, other=0.0)

    f_offs_m = tl.arange(0, 16)[:, None]
    f_offs_n = tl.arange(0, 16)[None, :]
    f_idx = f_offs_m * 16 + f_offs_n
    F_re = tl.load(F_re_ptr + f_idx)
    F_im = tl.load(F_im_ptr + f_idx)

    x_re = tl.reshape(x_re, (BLOCK_B, 16, 16))
    x_im = tl.reshape(x_im, (BLOCK_B, 16, 16))

    if STAGE_STOP > 0:

        x_re = tl.permute(x_re, (0, 2, 1))
        x_im = tl.permute(x_im, (0, 2, 1))


        x_re_flat = tl.reshape(x_re, (BLOCK_B * 16, 16))
        x_im_flat = tl.reshape(x_im, (BLOCK_B * 16, 16))

        x_re_flat, x_im_flat = _cdot(x_re_flat, x_im_flat, F_re, F_im)

        x_re = tl.reshape(x_re_flat, (BLOCK_B, 16, 16))
        x_im = tl.reshape(x_im_flat, (BLOCK_B, 16, 16))


        x_re = tl.permute(x_re, (0, 2, 1))
        x_im = tl.permute(x_im, (0, 2, 1))

        x_re = x_re.to(tl.float16)
        x_im = x_im.to(tl.float16)

    if STAGE_STOP > 1:

        tw_offs_m = tl.arange(0, 16)[:, None]
        tw_offs_n = tl.arange(0, 16)[None, :]
        tw_idx = 1 * 256 + tw_offs_n * 16 + tw_offs_m

        tw_re = tl.load(tw_re_ptr + tw_idx)
        tw_im = tl.load(tw_im_ptr + tw_idx)

        tw_re = tl.broadcast_to(tw_re[None, :, :], (BLOCK_B, 16, 16))
        tw_im = tl.broadcast_to(tw_im[None, :, :], (BLOCK_B, 16, 16))

        nx_re = x_re * tw_re - x_im * tw_im
        nx_im = x_re * tw_im + x_im * tw_re


        nx_re_flat = tl.reshape(nx_re, (BLOCK_B * 16, 16))
        nx_im_flat = tl.reshape(nx_im, (BLOCK_B * 16, 16))

        x_re_flat, x_im_flat = _cdot(nx_re_flat, nx_im_flat, F_re, F_im)

        x_re = tl.reshape(x_re_flat, (BLOCK_B, 16, 16))
        x_im = tl.reshape(x_im_flat, (BLOCK_B, 16, 16))


        x_re = tl.permute(x_re, (0, 2, 1))
        x_im = tl.permute(x_im, (0, 2, 1))

        x_re = x_re.to(tl.float16)
        x_im = x_im.to(tl.float16)

    x_re = tl.reshape(x_re, (BLOCK_B, 256))
    x_im = tl.reshape(x_im, (BLOCK_B, 256))

    if STORE_T:
        b_outer = offs_b // M
        m_inner = offs_b % M
        out_offs = b_outer[:, None] * (256 * M) + offs_n[None, :] * M + m_inner[:, None]
    else:
        out_offs = in_offs

    tl.store(y_re_ptr + out_offs, x_re, mask=mask_b)
    tl.store(y_im_ptr + out_offs, x_im, mask=mask_b)


# =============================================================================
# dft_kernel: padded length-R DFT for the small chunks (R in {2, 4, 8, 16})
# =============================================================================

@triton.jit
def dft_kernel(
    x_re_ptr, x_im_ptr,
    y_re_ptr, y_im_ptr,
    M_re_ptr, M_im_ptr,
    rows, M,
    R: tl.constexpr,
    BLOCK_B: tl.constexpr,
    STORE_T: tl.constexpr,
):
    pid = tl.program_id(0)
    offs_b = pid * BLOCK_B + tl.arange(0, BLOCK_B)

    offs_r_pad = tl.arange(0, 16)
    in_offs_pad = offs_b[:, None] * R + offs_r_pad[None, :]


    mask_pad = (offs_b[:, None] < rows) & (offs_r_pad[None, :] < R)

    x_re_pad = tl.load(x_re_ptr + in_offs_pad, mask=mask_pad, other=0.0)
    x_im_pad = tl.load(x_im_ptr + in_offs_pad, mask=mask_pad, other=0.0)

    f_offs_m = tl.arange(0, 16)[:, None]
    f_offs_n = tl.arange(0, 16)[None, :]
    f_idx = f_offs_m * 16 + f_offs_n

    M_re = tl.load(M_re_ptr + f_idx)
    M_im = tl.load(M_im_ptr + f_idx)

    out_re, out_im = _cdot(x_re_pad, x_im_pad, M_re, M_im)


    y_re = tl.reshape(out_re, (BLOCK_B, 16)).to(tl.float16)
    y_im = tl.reshape(out_im, (BLOCK_B, 16)).to(tl.float16)

    offs_r_out = tl.arange(0, 16)
    if STORE_T:
        b_outer = offs_b // M
        m_inner = offs_b % M
        out_offs = b_outer[:, None] * (R * M) + offs_r_out[None, :] * M + m_inner[:, None]
    else:
        out_offs = offs_b[:, None] * R + offs_r_out[None, :]


    mask_store = (offs_b[:, None] < rows) & (offs_r_out[None, :] < R)
    tl.store(y_re_ptr + out_offs, y_re, mask=mask_store)
    tl.store(y_im_ptr + out_offs, y_im, mask=mask_store)


# =============================================================================
# bailey_scale_kernel: elementwise w_N^{n1 kM} multiply with optional fused T2
# =============================================================================

@triton.jit
def bailey_scale_kernel(
    x_re_ptr, x_im_ptr,
    y_re_ptr, y_im_ptr,
    tw_re_ptr, tw_im_ptr,
    m0, M,
    BLOCK_M0: tl.constexpr,
    BLOCK_M: tl.constexpr,
    STORE_T: tl.constexpr,
):
    pid_m0 = tl.program_id(0)
    pid_M = tl.program_id(1)
    pid_row = tl.program_id(2)

    offs_m0 = pid_m0 * BLOCK_M0 + tl.arange(0, BLOCK_M0)
    offs_M = pid_M * BLOCK_M + tl.arange(0, BLOCK_M)

    mask = (offs_m0[:, None] < m0) & (offs_M[None, :] < M)

    in_offs = pid_row * (m0 * M) + offs_m0[:, None] * M + offs_M[None, :]
    tw_offs = offs_m0[:, None] * M + offs_M[None, :]

    v_re = tl.load(x_re_ptr + in_offs, mask=mask)
    v_im = tl.load(x_im_ptr + in_offs, mask=mask)

    t_re = tl.load(tw_re_ptr + tw_offs, mask=mask)
    t_im = tl.load(tw_im_ptr + tw_offs, mask=mask)


    v_re_f32 = v_re.to(tl.float32)
    v_im_f32 = v_im.to(tl.float32)
    t_re_f32 = t_re.to(tl.float32)
    t_im_f32 = t_im.to(tl.float32)

    out_re = (v_re_f32 * t_re_f32 - v_im_f32 * t_im_f32).to(tl.float16)
    out_im = (v_re_f32 * t_im_f32 + v_im_f32 * t_re_f32).to(tl.float16)

    if STORE_T:
        out_offs = pid_row * (M * m0) + offs_M[None, :] * m0 + offs_m0[:, None]
    else:
        out_offs = in_offs

    tl.store(y_re_ptr + out_offs, out_re, mask=mask)
    tl.store(y_im_ptr + out_offs, out_im, mask=mask)


# =============================================================================
# Thin launch wrappers -- GIVEN, do not edit
# =============================================================================

def _transpose(in_re, in_im, out_re, out_im, B, R, C):
    """Logical (B, R, C) -> (B, C, R) transpose, paired re/im."""
    grid = (triton.cdiv(R, TRANSPOSE_BLOCK), triton.cdiv(C, TRANSPOSE_BLOCK), B)
    transpose_kernel[grid](
        in_re, in_im, out_re, out_im, R, C,
        BLOCK_R=TRANSPOSE_BLOCK, BLOCK_C=TRANSPOSE_BLOCK,
    )

def _fft_chunk(in_re, in_im, out_re, out_im, rows, m, plan, M=1, store_t=False):
    """Length-m FFT over `rows` contiguous (rows, m) signals."""
    if m == 256:
        f4_plan = plan['f4_plan']
        f4_kernel_L2[(triton.cdiv(rows, F4_L2_BLOCK_B),)](
            in_re.view(rows, 256), in_im.view(rows, 256),
            out_re.view(rows, 256), out_im.view(rows, 256),
            f4_plan['F_re'], f4_plan['F_im'],
            f4_plan['tw_re'], f4_plan['tw_im'],
            rows, M,
            BLOCK_B=F4_L2_BLOCK_B, STAGE_STOP=f4_plan['L'], STORE_T=store_t,
            num_warps=4, num_stages=1,
        )
    else:
        M_re, M_im = plan['dft_mats'][m]
        dft_kernel[(triton.cdiv(rows, DFT_BLOCK_B),)](
            in_re.view(rows, m), in_im.view(rows, m),
            out_re.view(rows, m), out_im.view(rows, m),
            M_re, M_im, rows, M,
            R=m, BLOCK_B=DFT_BLOCK_B, STORE_T=store_t,
        )

def _scale(in_re, in_im, out_re, out_im, rows, m0, M, twr, twi, store_t=False):
    """Bailey scale over logical (rows, m0, M)."""
    grid = (triton.cdiv(m0, SCALE_BLOCK), triton.cdiv(M, SCALE_BLOCK), rows)
    bailey_scale_kernel[grid](
        in_re, in_im, out_re, out_im, twr, twi,
        m0, M, BLOCK_M0=SCALE_BLOCK, BLOCK_M=SCALE_BLOCK, STORE_T=store_t,
    )

def _lookup_tw(plan, m0, M, N_i):
    """Find the precomputed Bailey twiddle table for (m0, M, N_i) in plan['tw']."""
    for (a, b, n, tr, ti) in plan['tw']:
        if a == m0 and b == M and n == N_i:
            return tr, ti
    raise KeyError(f"no twiddle table for (m0={m0}, M={M}, N={N_i})")


# =============================================================================
# F3 pipeline: 4-step Bailey six-step (T1 -> F2-A -> T2 -> F2-B)
# =============================================================================

def f3_launch(in_re, in_im, out_re, out_im, mid_re, mid_im, plan, B):
    N1, N2 = plan['N1'], plan['N2']

    tw1_re, tw1_im, perm1 = plan['tw_re_n1'], plan['tw_im_n1'], plan['perm_n1']
    tw2_re, tw2_im, perm2 = plan['tw_re_n2'], plan['tw_im_n2'], plan['perm_n2']


    _transpose(in_re, in_im, mid_re, mid_im, B, N2, N1)


    f2_kernel[(B * N1,)](
        mid_re, mid_im, out_re, out_im,
        tw2_re, tw2_im, perm2,
        plan['bt_re'], plan['bt_im'],
        N1, N1 * N2, N2, plan['LOG2_N2'],
        BAILEY_EPILOGUE=True, STRIDED_STORE=False
    )


    _transpose(out_re, out_im, mid_re, mid_im, B, N1, N2)


    f2_kernel[(B * N2,)](
        mid_re, mid_im, out_re, out_im,
        tw1_re, tw1_im, perm1,
        tw1_re, tw1_im, # sentinels
        N2, N1 * N2, N1, plan['LOG2_N1'],
        BAILEY_EPILOGUE=False, STRIDED_STORE=True
    )


# =============================================================================
# F5 pipeline: 6-step Bailey at N1=N2=256 with F4 as inner FFT
# =============================================================================

def f5_launch(in_re, in_im, b0_re, b0_im, b1_re, b1_im, b2_re, b2_im, plan, B):
    N1, N2 = plan['N1'], plan['N2']


    _transpose(in_re, in_im, b0_re, b0_im, B, N2, N1)

    _fft_chunk(b0_re, b0_im, b1_re, b1_im, B * N1, N2, plan)

    _scale(b1_re, b1_im, b0_re, b0_im, B, N1, N2, plan['bt_re'], plan['bt_im'])

    _transpose(b0_re, b0_im, b1_re, b1_im, B, N1, N2)

    _fft_chunk(b1_re, b1_im, b2_re, b2_im, B * N2, N1, plan)

    _transpose(b2_re, b2_im, b0_re, b0_im, B, N2, N1)


# =============================================================================
# F6 / F7 recursion
# =============================================================================

def _f6_rec(cur_re, cur_im, rows, chunks, plan, cyc):
    if len(chunks) == 1:
        out_re, out_im = cyc.next()
        _fft_chunk(cur_re, cur_im, out_re, out_im, rows, chunks[0], plan)
        return out_re, out_im

    m0 = chunks[0]
    M = math.prod(chunks[1:])
    N_i = m0 * M


    mid_re, mid_im = cyc.next()
    _transpose(cur_re, cur_im, mid_re, mid_im, rows, M, m0)


    rec_re, rec_im = _f6_rec(mid_re, mid_im, rows * m0, chunks[1:], plan, cyc)


    sc_re, sc_im = cyc.next()
    tw_re, tw_im = _lookup_tw(plan, m0, M, N_i)
    _scale(rec_re, rec_im, sc_re, sc_im, rows, m0, M, tw_re, tw_im, store_t=False)


    t2_re, t2_im = cyc.next()
    _transpose(sc_re, sc_im, t2_re, t2_im, rows, m0, M)


    fft_re, fft_im = cyc.next()
    _fft_chunk(t2_re, t2_im, fft_re, fft_im, rows * M, m0, plan, store_t=False)


    out_re, out_im = cyc.next()
    _transpose(fft_re, fft_im, out_re, out_im, rows, M, m0)

    return out_re, out_im


def _f7_rec(cur_re, cur_im, rows, chunks, plan, cyc):
    if len(chunks) == 1:
        out_re, out_im = cyc.next()
        _fft_chunk(cur_re, cur_im, out_re, out_im, rows, chunks[0], plan)
        return out_re, out_im

    m0 = chunks[0]
    M = math.prod(chunks[1:])
    N_i = m0 * M


    mid_re, mid_im = cyc.next()
    _transpose(cur_re, cur_im, mid_re, mid_im, rows, M, m0)


    rec_re, rec_im = _f7_rec(mid_re, mid_im, rows * m0, chunks[1:], plan, cyc)


    sc_re, sc_im = cyc.next()
    tw_re, tw_im = _lookup_tw(plan, m0, M, N_i)
    _scale(rec_re, rec_im, sc_re, sc_im, rows, m0, M, tw_re, tw_im, store_t=True)


    out_re, out_im = cyc.next()
    _fft_chunk(sc_re, sc_im, out_re, out_im, rows * M, m0, plan, M=M, store_t=True)

    return out_re, out_im

Overwriting kernels.py


## Cell 5: Check the Twiddle File using the twiddle_check.py file

In [ ]:
!python twiddle_check.py

## Cell 6: Check the kernals by using sanity_check.py file

In [ ]:
!python sanity_check.py

# Core Theory

## The Bottleneck of the Naive DFT

The standard DFT is an $O(N^2)$ algorithm. For every output element, you have to look at every input element. On GPUs, this is terrible because it wastes massive amounts of compute.


## Cooley-Tukey & The "Butterfly"

Cooley-Tukey is the algorithm that reduces the complexity to $O(N \log N)$. It does this by recursively splitting the DFT into smaller and smaller DFTs (e.g., splitting a length-N transform into two length-N/2 transforms).

Bit-Reversal: Because we recursively split the array into even and odd indices, the data ends up physically scrambled. To compute the FFT efficiently in-place, we must first load the data using "bit-reversed" indices.

The Butterfly: The fundamental compute step combining two smaller sub-problems. It involves a complex multiplication (by a Twiddle Factor) and an addition/subtraction.

## Twiddle Factors ($W_N^k$)

When you break a large FFT into smaller pieces, you can't just glue the answers back together. Because you shifted the position of the data in time, you have to shift its phase in frequency. Twiddle factors are these complex phase rotations (sines and cosines) applied during the butterfly steps.

## Bailey's 6-Step Matrix Factorization

As $N$ gets huge, you can no longer fit the array into the GPU's ultra-fast SRAM (Shared Memory/Registers). If you try to do standard Cooley-Tukey in Global Memory (VRAM), the memory access patterns become huge, strided jumps, which crushes memory bandwidth.

*Bailey's Solution:* Treat a 1D array of length $N$ as a 2D matrix of size $N_1 \times N_2$.

The 6 Steps:
1. Transpose the matrix.
2. Do small FFTs on the rows.
3. Multiply by Bailey Twiddle factors.
4. Transpose again.
5. Do small FFTs on the rows again.
6. Transpose back.

*Why it works?*: By transposing the data, the sub-FFTs always read contiguous blocks of memory, keeping the GPU's memory bandwidth fully saturated.

## Tensor Cores & tcFFT (Radix-16)

Standard Cooley-Tukey (Radix-2) uses scalar math (1 multiply, 1 add). Modern GPUs have Tensor Cores, massive hardware units dedicated solely to matrix multiplication (Matmul). To use them, we switch to Radix-16 tcFFT, converting the FFT into a sequence of small $16 \times 16$ matrix multiplications to max out the GPU's FLOPs.

# The Kernal Breakdown

## F1: The Baseline (Dense Matmul)

**What it is:** The mathematical definition of a DFT represented as a single dense Matrix Multiplication: $Y = X \times W^T$.

*Implementation:* Complex numbers mean one multiplication becomes four real multiplications: $(A+Bi)(C+Di) = (AC-BD) + (AD+BC)i$. We use four `tl.dot()` calls to compute this.

**Why we do it:** It verifies our complex matmul helper `_cdot` works, but it's $O(N^2)$, so it scales horribly.

## F2: Radix-2 Cooley-Tukey

**What it is:** The classic $O(N \log N)$ FFT algorithm running entirely in ultra-fast GPU registers.

***Implementation Mechanics:***


1.   **Bit-reversal load:** It reads from global memory using the pre-computed `perm_ptr` to load data in bit-reversed order.
2.   **In-Register Loop:** For `log2(N)` stages, it does the butterfly. It uses `tl.gather` with an XOR mask (`partner = offs ^ half_step`) to find the partner register
3. **No SRAM:** Triton handles the register allocation. Because there is no shared memory sync (`tl.debug_barrier`), it is incredibly fast, but structurally limited by the max number of registers a GPU thread block can hold (starts failing past N ~ 16,384).



## F3: Bailey 6-Step Pipeline

**What it is:** Breaks a larger FFT into two smaller F2 FFTs to bypass F2's register limits.

***Implementation Mechanics:*** * Instead of doing 6 full trips to global memory, F3 uses Kernel Fusion to squash it down to 4 steps.

1.  F2-A Fusion: The `bailey_scale_kernel` (Step 3) is fused directly into the end of the first F2 run using the `BAILEY_EPILOGUE=True` flag.

2.  F2-B Fusion: The final Transpose (Step 6) is skipped by telling the second F2 run to write its output in a transposed layout using `STRIDED_STORE=True`.

## F4: tcFFT Radix-16

**What it is:** Shifts the math from scalar operations (ALUs) to Tensor Cores using `tl.dot()`. Used specifically for $N=256$.

***Implementation Mechanics:***

Views length-256 as a $16 \times 16$ grid ($16^2 = 256$).

Stage 0: Reshapes to 3D, permutes the axis to the end, flattens to 2D, and uses `tl.dot()` against the $F_{16}$ Fourier matrix to transform the first base-16 digit.

Stage 1: Swaps the axes, broadcasts and multiplies the Radix-16 twiddle factors, flattens, and uses `tl.dot()` to transform the second base-16 digit.

This kernel is extremely picky about shape dimensions because tl.dot mathematically requires strict `M x K` and `K x N` 2D boundaries.

## F5: Bailey + F4

**What it is:** Bridging the gap. It is identical in structure to F3, but it applies Bailey's 6-step factorization for $N = 65,536$ by using the tensor-core-accelerated F4 as the sub-FFT instead of F2.

***Implementation:*** It literally just calls `_transpose`, `_fft_chunk` (which maps to F4), `_scale`, etc., in sequence. No fancy fusion here, just proving the logic works at scale.

## F6: Recursive Bailey

**What it is:** A recursive python loop that can handle any arbitrarily massive size of $N$ by chunking it down.

***Implementation Mechanics:*** It uses `f6_factor` to greedily break $N$ into chunks of 256 (for F4), then 16 (for padded F1), then leftovers. It builds the 6-step pipeline recursively, calling itself until it hits a leaf node (a base chunk), at which point it runs the base FFT.

*Problem:* It results in an enormous number of intermediate `_transpose` and `_scale` kernel launches, heavily wasting Global Memory bandwidth.

## F7: Fused Recursive Bailey

**What it is:** The final boss. F7 implements the exact same recursive math as F6, but heavily optimizes memory bandwidth via Kernel Fusion.

***Implementation Mechanics:***  Every non-leaf level of Bailey has a `Scale` followed by a `Transpose (T2)`. F7 fuses this: it tells the scale kernel to write its output to memory in a transposed pattern (`STORE_T=True`), entirely eliminating the T2 kernel launch.

1.  It does the same for the inner `FFT` and `Transpose (T3)`, saving yet another massive trip to VRAM.

2.  Because the floating-point math happens in the exact same sequence as F6 (just stored to memory differently), F7 guarantees bitwise equality with F6 while running much faster.